# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

### Bitsandbytes:
bitsandbytes is a lightweight Python wrapper for CUDA custom functions, primarily used to optimize large language models (LLMs) by reducing memory usage through 8-bit and 4-bit quantization. It enables efficient inference (LLM.int8()) and training (QLoRA) on limited hardware, supporting NVIDIA CUDA 11.8 - 13.0 and offering experimental AMD GPU support.

Key Features & Capabilities:
- 4-bit and 8-bit Quantization: Enables running large models (like LLaMA, Mistral) on smaller GPUs without significant performance loss.
- QLoRA (Quantized LoRA): Allows fine-tuning of 65B+ parameter models on a single GPU by loading base models in 4-bit.
- 8-bit Optimizers: Reduces memory footprint for optimization states (e.g., Adam Optimizer), freeing up memory for larger batch sizes.
- Stable Embedding Layers: Improves stability during training with optimized embeddings.

Main Use Cases:
- Inference: Running quantized models to reduce VRAM requirements.
- Fine-tuning: Training large models using low-resource setups.
- 8-bit Matrix Multiplication: Faster and more memory-efficient computations.

### Accelerate:
Hugging Face Accelerate is an open-source library that allows PyTorch users to run their training scripts on any distributed configuration (single GPU, multi-GPU, TPU, etc.) with minimal code changes.

Key Feature:
- It abstracts the boilerplate code required for distributed training and mixed precision (FP16, BF16), leaving the core training loop unchanged.

Core Components:
- The Accelerator Class: The main object used to prepare models, optimizers, and data loaders for any environment.
- CLI Launcher: Use accelerate config to set up your environment and accelerate launch to execute scripts.

Big Model Inference: It includes utilities for loading and running extremely large models that do not fit in a single device's memory.

In [2]:
# Installing Libraries:

# Run code line below if 'bitsandbytes' and 'accelerate' libraries are not already installed:
#!pip install bitsandbytes accelerate

In [4]:
# Logging in to Hugging Face:

import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv(override= True, verbose= True)

hf_token = os.getenv("HF_TOKEN")

if hf_token and hf_token.startswith('hf_'):
    print('Hugging Face Token Found.')
    login(token= hf_token, add_to_git_credential= True)
    print('Login to Hugging Face Successful.')
else:
    print('Hugging Face Token Not Found.')

Hugging Face Token Found.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login to Hugging Face Successful.


In [5]:
# Importing Necessary Libraries:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [6]:
# Defining Models:

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

`Note: All models defined above are intentionally choosen because of their small size to download faster and make it possible to run them on local GPU.`

In [7]:
# Defining Messages:

messages = [{'role': 'user', 'content': 'Tell a joke for a room of Data Scientists.'}]

In [8]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_quant_type= 'nf4'
)

In [11]:
# Tokenizer:

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors= 'pt').to('cuda')

print(inputs)

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2839,   5186,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116,     13, 128009]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [12]:
# The LLAMA Model:
# Note: This will download the whole LLAMA model (defined above) on local storage, so, it will take some time to run:
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map= 'auto', quantization_config= quant_config)

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [15]:
# Memory used by LLAMA model:
print(f'Memory Footprint of LLAMA Model (in Bytes): {model.get_memory_footprint()}')
print(f'Memory Footprint of LLAMA Model (in KBs): {model.get_memory_footprint() / 1024}')
print(f'Memory Footprint of LLAMA Model (in MBs): {model.get_memory_footprint() / (1024*1024)}')

Memory Footprint of LLAMA Model (in Bytes): 1012011264
Memory Footprint of LLAMA Model (in KBs): 988292.25
Memory Footprint of LLAMA Model (in MBs): 965.129150390625


### Looking under the hood at the Transformer model:

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, this also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [20]:
# Execute this cell and look at what gets printed; investigate the layers

model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm